# Análise Exploratória dos Dados Limpos
## ProScore Analytics — CreditGuard AI

**Disciplina:** Introdução à Ciência de Dados  
**Dataset:** Home Credit Default Risk (Kaggle)  
**Arquivo analisado:** `Dados/clean_data.csv` (saída da etapa de sanitização realizada em `data_preparation.ipynb`)

---

### Objetivo

Este notebook apresenta a análise exploratória dos dados após a etapa de sanitização para o projeto **CreditGuard AI**, sistema de predição de inadimplência em instituições financeiras. O objetivo é compreender a estrutura, qualidade e distribuições dos dados antes da etapa de modelagem preditiva.

O conjunto de dados contém informações de solicitações de crédito do **Home Credit Default Risk** (Kaggle), onde a variável alvo `TARGET = 1` indica que o cliente ficou inadimplente no empréstimo.

---

### Roteiro da Análise

1. Carregamento dos dados limpos  
2. Dimensionalidade e estrutura  
3. Tipos de variáveis  
4. Análise de valores ausentes  
5. Estatísticas descritivas  
6. Distribuição da variável TARGET  
7. Análise univariada das principais variáveis  
8. Matriz de correlação  
9. Insights relevantes para risco de crédito  
10. Conclusões

## 1. Carregamento dos Dados

Os dados carregados nesta análise correspondem ao arquivo `clean_data.csv`, gerado pela etapa de sanitização descrita em `data_preparation.ipynb`. Esse arquivo preserva a estrutura original da base, mas com tratamentos aplicados: remoção de duplicatas, padronização de tipos e tratamento de valores especiais identificados na análise de qualidade dos dados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 50)

df = pd.read_csv('../Dados/clean_data.csv')

print(f"Dataset carregado com sucesso.")
print(f"  Linhas  : {df.shape[0]:,}")
print(f"  Colunas : {df.shape[1]}")
df.head()

## 2. Dimensionalidade e Estrutura dos Dados

Análise inicial da estrutura do dataset: número de registros, número de variáveis e visão geral das colunas disponíveis.

In [ ]:
print("=" * 50)
print("DIMENSOES DO DATASET")
print("=" * 50)
print(f"  Registros : {df.shape[0]:,}")
print(f"  Variaveis : {df.shape[1]}")
print()
print(f"Primeiras 5 colunas : {list(df.columns[:5])}")
print(f"Ultimas  5 colunas  : {list(df.columns[-5:])}")
print()
print("Lista completa de variaveis:")
print(df.columns.tolist())

> **Interpretação:** O dataset possui registros de solicitações de crédito com dezenas de variáveis cobrindo perfil socioeconômico, situação habitacional, documentação apresentada, histórico de consultas ao bureau de crédito e indicadores externos de score. A variável alvo `TARGET` indica se o cliente se tornou inadimplente.

## 3. Tipos de Variáveis

A identificação correta dos tipos de variáveis é essencial para escolher as técnicas adequadas de análise e para definir o tratamento de encoding na etapa de preparação dos dados.

In [ ]:
df.info()

In [ ]:
num_num = df.select_dtypes(include=[np.number]).shape[1]
num_cat = df.select_dtypes(include='object').shape[1]

print(f"Variaveis numericas   : {num_num}")
print(f"Variaveis categoricas : {num_cat}")
print()
print("Variaveis categoricas e suas cardinalidades:")
display(df.select_dtypes(include='object').nunique().rename('n_categorias').to_frame())
print()
print("Cardinalidade geral (top 30 menores):")
display(df.nunique().sort_values().head(30).rename('n_valores_unicos').to_frame())

> **Interpretação:** A base é majoritariamente numérica. As variáveis categóricas são de baixa cardinalidade (binárias e poucas categorias), o que facilita o encoding. Variáveis como `NAME_CONTRACT_TYPE`, `CODE_GENDER` e `FLAG_OWN_CAR` possuem apenas 2–3 categorias. Variáveis de identificação (`SK_ID_CURR`) devem ser removidas antes da modelagem.

## 4. Análise de Valores Ausentes

A presença de valores ausentes é relevante tanto para a qualidade dos dados quanto como sinal preditivo: a ausência de um campo pode indicar que o cliente não possui aquele tipo de histórico — por exemplo, ausência em `EXT_SOURCE_1` pode indicar que o bureau não possui score cadastrado para o cliente.

In [ ]:
missing_abs = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing_abs / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Ausentes': missing_abs, 'Ausentes (%)': missing_pct})
print("Top 20 variaveis com mais valores ausentes:")
display(missing_df[missing_df['Ausentes'] > 0].head(20))

In [ ]:
top_missing = missing_pct[missing_pct > 0].sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_missing.index, top_missing.values, color='steelblue', edgecolor='white')
ax.axvline(x=30, color='red', linestyle='--', linewidth=1.2, label='Limite 30%')
ax.axvline(x=50, color='orange', linestyle='--', linewidth=1.2, label='Limite 50%')
ax.set_xlabel('% de Valores Ausentes')
ax.set_title('Top 20 Variaveis com Maior Proporcao de Valores Ausentes')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()

> **Interpretação:** As variáveis de score externo (`EXT_SOURCE_1`, `EXT_SOURCE_3`) possuem alta proporção de missings — mas a ausência não é um erro de coleta: indica que o cliente não possui cadastro no bureau externo, o que por si só é informação preditiva. Por essa razão, foram criadas flags de ausência (`EXT_SOURCE_1_MISSING`, `EXT_SOURCE_3_MISSING`) na etapa de preparação dos dados.

## 5. Estatísticas Descritivas

A análise das estatísticas descritivas permite identificar a magnitude, dispersão e eventuais outliers das variáveis numéricas, além de orientar decisões de transformação para a modelagem.

In [ ]:
df.describe().T

In [ ]:
key_cols = ['TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
            'DAYS_BIRTH', 'DAYS_EMPLOYED', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
key_cols_exist = [c for c in key_cols if c in df.columns]
df[key_cols_exist].describe().T

> **Interpretação:**
> - `AMT_INCOME_TOTAL`: Renda com grande amplitude — média acima da mediana indica distribuição assimétrica à direita.
> - `AMT_CREDIT`: Valor do crédito varia amplamente; a relação crédito/renda é um indicador-chave de risco.
> - `DAYS_BIRTH`: Valores negativos representam dias antes da data de referência (idade = `|DAYS_BIRTH| / 365`).
> - `DAYS_EMPLOYED = 365243`: Código especial presente em ~18% dos registros, representando clientes inativos/aposentados. Tratado como categoria especial, **não como outlier**.
> - `EXT_SOURCE_1/2/3`: Scores externos normalizados (0–1). Quanto maior o score, menor o risco de inadimplência.

## 6. Distribuição da Variável TARGET

`TARGET` é a variável dependente do modelo: `1` indica que o cliente ficou inadimplente, `0` indica adimplência. O balanceamento entre as classes impacta diretamente a estratégia de modelagem preditiva.

In [ ]:
target_counts = df['TARGET'].value_counts()
target_pct    = df['TARGET'].value_counts(normalize=True).mul(100).round(2)

print("Distribuicao absoluta (TARGET):")
print(target_counts.rename({0: 'Adimplente (0)', 1: 'Inadimplente (1)'}).to_string())
print()
print("Distribuicao relativa (%)")
print(target_pct.rename({0: 'Adimplente (0)', 1: 'Inadimplente (1)'}).to_string())
print()
print(f"Taxa de inadimplencia: {target_pct.get(1, 0):.2f}%")
print(f"Razao de desbalanceamento: {target_counts.get(0,0) / max(target_counts.get(1,1), 1):.1f}:1")

In [ ]:
counts = df['TARGET'].value_counts()
pcts   = df['TARGET'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(['Adimplente (0)', 'Inadimplente (1)'],
            [counts.get(0, 0), counts.get(1, 0)],
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Contagem por Classe (TARGET)')
axes[0].set_ylabel('Numero de Clientes')
for i, v in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    axes[0].text(i, v + 800, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie([pcts.get(0, 0), pcts.get(1, 0)],
            labels=['Adimplente (0)', 'Inadimplente (1)'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Proporcao por Classe (TARGET)')

plt.suptitle('Distribuicao da Variavel TARGET', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Interpretação:** O dataset apresenta **forte desbalanceamento de classes**: aproximadamente 91,9% dos clientes são adimplentes e apenas 8,1% são inadimplentes. Esse padrão é típico de problemas de crédito. Implicações para a modelagem:
> - Um modelo trivial (classifica todos como adimplentes) teria ~92% de acurácia — métrica inadequada.
> - A métrica prioritária é o **Recall para a classe 1**: minimizar falsos negativos (inadimplentes classificados como bons pagadores).
> - Estratégia adotada: XGBoost com `scale_pos_weight ≈ 11.38` para compensar o desbalanceamento.

## 7. Análise Univariada das Principais Variáveis

Análise das distribuições individuais das variáveis numéricas mais relevantes para risco de crédito.

In [ ]:
ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df.columns]
print("Estatisticas dos Scores Externos (EXT_SOURCE):")
display(df[ext_cols].describe().T)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Idade (DAYS_BIRTH -> anos)
if 'DAYS_BIRTH' in df.columns:
    idade = df['DAYS_BIRTH'].abs() / 365
    axes[0].hist(idade.dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].set_title('Distribuicao de Idade (anos)')
    axes[0].set_xlabel('Idade')
    axes[0].set_ylabel('Frequencia')

variaveis = [
    ('AMT_INCOME_TOTAL', 'Renda Total'),
    ('AMT_CREDIT',       'Valor do Credito'),
    ('AMT_ANNUITY',      'Anuidade'),
    ('EXT_SOURCE_2',     'EXT_SOURCE_2'),
    ('EXT_SOURCE_3',     'EXT_SOURCE_3'),
]

for idx, (col, label) in enumerate(variaveis, start=1):
    if col not in df.columns or idx >= len(axes):
        continue
    data = df[col].dropna()
    p99 = data.quantile(0.99)
    data = data[data <= p99]
    axes[idx].hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[idx].set_title(f'Distribuicao: {label}')
    axes[idx].set_xlabel(label)
    axes[idx].set_ylabel('Frequencia')

plt.suptitle('Analise Univariada - Principais Variaveis Numericas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Interpretação:**
> - **Renda e Crédito:** Distribuições assimétricas à direita — a maioria dos clientes concentra-se em faixas mais baixas, com poucos clientes de alta renda/crédito. Isso justifica o uso de log-transformação em modelos lineares.
> - **Idade:** Distribuição aproximadamente normal entre 20 e 70 anos, com concentração entre 30 e 50 anos. Clientes mais jovens tendem a apresentar maior risco de inadimplência.
> - **EXT_SOURCE_2/3:** Scores externos com distribuições distintas. EXT_SOURCE_2 é mais uniforme; EXT_SOURCE_3 é mais concentrada em valores baixos-médios. Ambos são preditores negativos de inadimplência.

## 8. Matriz de Correlação

A análise de correlação entre variáveis numéricas ajuda a identificar relações lineares e possíveis problemas de multicolinearidade que podem afetar modelos lineares.

In [ ]:
corr_cols = [c for c in [
    'TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CNT_CHILDREN',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
] if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax
)
ax.set_title('Matriz de Correlacao - Principais Variaveis Numericas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelacao com TARGET (ordenado):")
print(corr_matrix['TARGET'].drop('TARGET').sort_values())

> **Interpretação:**
> - **EXT_SOURCE_2 e EXT_SOURCE_3** apresentam correlação negativa com TARGET: clientes com scores externos maiores têm menor probabilidade de inadimplência. São as variáveis mais preditivas disponíveis.
> - **DAYS_BIRTH (idade)** correlaciona-se negativamente com TARGET: clientes mais velhos são menos propensos à inadimplência.
> - **AMT_CREDIT e AMT_ANNUITY** possuem alta correlação entre si (relação estrutural: anuidade é derivada do crédito), o que deve ser considerado na seleção de features.
> - **CNT_CHILDREN** tem correlação positiva leve com TARGET: mais filhos pode indicar maior comprometimento de renda.

## 9. Insights Relevantes para Risco de Crédito

Análise bivariada entre a variável TARGET e as principais variáveis preditivas para quantificar os padrões de inadimplência observados nos dados.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. EXT_SOURCE_2 por TARGET
if 'EXT_SOURCE_2' in df.columns:
    for tv, color, label in [(0, 'steelblue', 'Adimplente (0)'), (1, 'tomato', 'Inadimplente (1)')]:
        s = df[df['TARGET'] == tv]['EXT_SOURCE_2'].dropna()
        axes[0, 0].hist(s, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[0, 0].set_title('EXT_SOURCE_2 por TARGET')
    axes[0, 0].set_xlabel('EXT_SOURCE_2')
    axes[0, 0].legend()

# 2. Idade por TARGET
if 'DAYS_BIRTH' in df.columns:
    for tv, color, label in [(0, 'steelblue', 'Adimplente (0)'), (1, 'tomato', 'Inadimplente (1)')]:
        s = (df[df['TARGET'] == tv]['DAYS_BIRTH'].abs() / 365).dropna()
        axes[0, 1].hist(s, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[0, 1].set_title('Distribuicao de Idade por TARGET')
    axes[0, 1].set_xlabel('Idade (anos)')
    axes[0, 1].legend()

# 3. Taxa de inadimplencia por faixa de credito
if 'AMT_CREDIT' in df.columns:
    df_c = df[df['AMT_CREDIT'] <= df['AMT_CREDIT'].quantile(0.99)].copy()
    df_c['faixa_credito'] = pd.cut(df_c['AMT_CREDIT'], bins=5,
                                    labels=['Muito Baixo','Baixo','Medio','Alto','Muito Alto'])
    taxa = df_c.groupby('faixa_credito', observed=True)['TARGET'].mean() * 100
    axes[1, 0].bar(taxa.index, taxa.values, color='steelblue', edgecolor='white')
    axes[1, 0].set_title('Taxa de Inadimplencia por Faixa de Credito')
    axes[1, 0].set_ylabel('Taxa de Inadimplencia (%)')
    axes[1, 0].set_xlabel('Faixa de Credito')

# 4. Taxa por numero de filhos
if 'CNT_CHILDREN' in df.columns:
    filhos_default = df[df['CNT_CHILDREN'] <= 5].groupby('CNT_CHILDREN')['TARGET'].mean() * 100
    axes[1, 1].bar(filhos_default.index.astype(str), filhos_default.values,
                   color='steelblue', edgecolor='white')
    axes[1, 1].set_title('Taxa de Inadimplencia por N de Filhos')
    axes[1, 1].set_ylabel('Taxa de Inadimplencia (%)')
    axes[1, 1].set_xlabel('Numero de Filhos')

plt.suptitle('Insights - Relacao das Variaveis com TARGET (Inadimplencia)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Insights identificados:**
> 1. **EXT_SOURCE_2 vs TARGET:** Clientes inadimplentes concentram-se em valores baixos de EXT_SOURCE_2, confirmando que scores externos baixos são sinal de risco. A separação entre as distribuições é clara — EXT_SOURCE_2 é um preditor forte.
> 2. **Idade vs TARGET:** Clientes mais jovens (20–35 anos) apresentam maior taxa de inadimplência. Clientes acima de 50 anos exibem comportamento de pagamento muito mais estável.
> 3. **Valor do crédito vs TARGET:** A relação não é monotônica — créditos de valor muito baixo ou médio tendem a ter taxas de inadimplência ligeiramente maiores, possivelmente associados a clientes sem histórico creditício estabelecido.
> 4. **Número de filhos vs TARGET:** Clientes com filhos apresentam taxa de inadimplência marginalmente maior, sugerindo maior comprometimento de renda domiciliar com despesas familiares.

## 10. Conclusões da Exploração

---

### Principais Conclusões

A análise exploratória dos dados limpos do projeto CreditGuard AI revelou os seguintes pontos:

#### Qualidade dos Dados
- O dataset possui estrutura rica com variáveis demográficas, financeiras e de histórico de crédito.
- Valores ausentes estão concentrados em variáveis de bureau externo (`EXT_SOURCE_1`, `EXT_SOURCE_3`) e variáveis imobiliárias — a ausência é estrutural e preditiva, não erro de coleta.
- O valor especial `DAYS_EMPLOYED = 365243` representa clientes inativos/aposentados e foi preservado como categoria distinta (taxa de inadimplência diferenciada: 5,4% vs 8,66% dos demais).

#### Perfil do Desbalanceamento
- Taxa de inadimplência de aproximadamente **8,07%**, gerando desbalanceamento 91,9:8,1 entre classes.
- Esse desbalanceamento exige estratégias específicas: `scale_pos_weight`, métricas orientadas a Recall e avaliação por ROC-AUC.

#### Variáveis mais Relevantes para Risco de Crédito

| Variável | Relação com Inadimplência |
|---|---|
| `EXT_SOURCE_2` / `EXT_SOURCE_3` | Correlação negativa forte — score alto = menor risco |
| `DAYS_BIRTH` (idade) | Clientes jovens apresentam maior inadimplência |
| `AMT_CREDIT` | Não linear — faixas extremas têm comportamentos distintos |
| `CNT_CHILDREN` | Leve aumento de risco com mais filhos |
| `DAYS_EMPLOYED = 365243` | Clientes inativos: taxa diferenciada (5,4% vs 8,66%) |

#### Decisões para a Etapa de Modelagem
1. Usar `EXT_SOURCE_1/2/3` e `DAYS_BIRTH` como features prioritárias.
2. Criar flags de ausência (`EXT_SOURCE_1_MISSING`, `EXT_SOURCE_3_MISSING`) para preservar o valor preditivo dos nulos.
3. Configurar `scale_pos_weight` no XGBoost para compensar o desbalanceamento.
4. Priorizar **Recall** como métrica principal de avaliação do modelo.
5. Manter `DAYS_EMPLOYED = 365243` como categoria especial, sem substituição por mediana.

---
*Análise concluída. Próxima etapa: `Model/evaluation.ipynb` — Treinamento e avaliação dos modelos preditivos.*